# Stock Panel Data Analysis

30 S&P 500 stocks · 36 months (2022–2024) · 1,080 rows · 13 columns

**Signals:** momentum, lagged_return, marketcap_billions, pb (price-to-book), roe, grossmargin, assetturnover, gp_to_assets, asset_growth  
**Outcome:** monthly return

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='tab10')

# ── Load data ──────────────────────────────────────────────────────────
df = pd.read_excel('slides/files/stock_panel.xlsx')
df['month'] = pd.to_datetime(df['month'])
df['year'] = df['month'].dt.year
df = df.sort_values(['ticker', 'month']).reset_index(drop=True)
print(df.shape)
df.head()

---
## 1. Data Overview

In [ ]:
# Table 1 — Summary statistics for numeric columns
num_cols = ['return','momentum','lagged_return','marketcap_billions',
            'pb','roe','grossmargin','assetturnover','gp_to_assets','asset_growth']
df[num_cols].describe().round(3)

In [ ]:
# Table 2 — Missing value counts
missing = df[num_cols].isna().sum().rename('missing').to_frame()
missing['pct'] = (missing['missing'] / len(df) * 100).round(1)
missing

---
## 2. Return Distributions

In [ ]:
# Figure 1 — Filled density plot of monthly returns
fig, ax = plt.subplots()
x = np.linspace(df['return'].min() - 0.01, df['return'].max() + 0.01, 300)
kde = stats.gaussian_kde(df['return'].dropna())
y = kde(x)
ax.fill_between(x, y, alpha=0.4, color='steelblue')
ax.plot(x, y, color='steelblue', lw=2)
ax.axvline(0, color='black', lw=0.8, ls='--')
ax.axvline(df['return'].mean(), color='crimson', lw=1.5, ls='--', label=f"Mean = {df['return'].mean():.3f}")
ax.set_xlabel('Monthly Return'); ax.set_ylabel('Density')
ax.set_title('Distribution of Monthly Returns (all stocks, all months)')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Figure 2 — Filled density plots by sector (overlaid)
sectors = sorted(df['sector'].unique())
palette = sns.color_palette('tab10', len(sectors))
fig, ax = plt.subplots(figsize=(11, 5))
for sec, col in zip(sectors, palette):
    vals = df.loc[df['sector'] == sec, 'return'].dropna()
    kde = stats.gaussian_kde(vals)
    x = np.linspace(-0.35, 0.40, 300)
    y = kde(x)
    ax.fill_between(x, y, alpha=0.15, color=col)
    ax.plot(x, y, color=col, lw=1.5, label=sec)
ax.axvline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('Monthly Return'); ax.set_ylabel('Density')
ax.set_title('Return Distribution by Sector')
ax.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

In [ ]:
# Figure 3 — Box plots of monthly returns by sector
order = df.groupby('sector')['return'].median().sort_values().index
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df, x='sector', y='return', order=order,
            palette='tab10', width=0.5, fliersize=3, ax=ax)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel(''); ax.set_ylabel('Monthly Return')
ax.set_title('Monthly Returns by Sector')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.show()

In [ ]:
# Figure 4 — Violin plots of monthly returns by sector
fig, ax = plt.subplots(figsize=(11, 5))
sns.violinplot(data=df, x='sector', y='return', order=order,
               palette='tab10', inner='quartile', ax=ax)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel(''); ax.set_ylabel('Monthly Return')
ax.set_title('Return Distributions by Sector (Violin)')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.show()

---
## 3. Sector Analysis

In [ ]:
# Table 3 — Return statistics by sector
sector_stats = df.groupby('sector')['return'].agg(
    Mean='mean', Median='median', Std='std',
    Min='min', Max='max', Count='count'
).round(4).sort_values('Mean', ascending=False)
sector_stats

In [ ]:
# Figure 5 — Bar chart: mean monthly return by sector with error bars
sec_mean = df.groupby('sector')['return'].agg(['mean','sem']).sort_values('mean')
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d62728' if v < 0 else '#1f77b4' for v in sec_mean['mean']]
ax.barh(sec_mean.index, sec_mean['mean'],
        xerr=sec_mean['sem'] * 1.96, color=colors,
        capsize=4, edgecolor='white', height=0.6)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Mean Monthly Return')
ax.set_title('Mean Monthly Return by Sector (±95% CI)')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
plt.tight_layout(); plt.show()

In [ ]:
# Figure 6 — Pie chart: average market cap by sector
sec_cap = df.groupby('sector')['marketcap_billions'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 7))
wedges, texts, autotexts = ax.pie(
    sec_cap.values, labels=sec_cap.index,
    autopct='%1.1f%%', startangle=140,
    colors=sns.color_palette('tab10', len(sec_cap)),
    pctdistance=0.82, wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for t in autotexts: t.set_fontsize(8)
ax.set_title('Share of Average Market Cap by Sector')
plt.tight_layout(); plt.show()

In [ ]:
# Figure 7 — Stacked bar chart: average return by sector by year
yr_sec = df.groupby(['year','sector'])['return'].mean().unstack()
yr_sec_pct = yr_sec * 100
fig, ax = plt.subplots(figsize=(10, 5))
yr_sec_pct.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Year'); ax.set_ylabel('Mean Monthly Return (%)')
ax.set_title('Average Monthly Return by Sector and Year')
ax.legend(title='Sector', fontsize=8, loc='upper right', ncol=2)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# Figure 8 — Heatmap: mean return per sector per year
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(yr_sec.T * 100, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Mean Monthly Return (%)'})
ax.set_title('Mean Monthly Return (%) by Sector and Year')
ax.set_xlabel('Year'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

---
## 4. Time-Series Portfolio Returns

In [ ]:
# Equal-weighted portfolio return each month
ew = df.groupby('month')['return'].mean().rename('ew_return')
ew_cum = (1 + ew).cumprod() - 1

# Figure 9 — Equal-weighted portfolio cumulative return
fig, axes = plt.subplots(2, 1, figsize=(11, 8))

axes[0].bar(ew.index, ew.values * 100,
            color=['#d62728' if v < 0 else '#1f77b4' for v in ew],
            width=20)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('Equal-Weighted Portfolio: Monthly Returns')
axes[0].set_ylabel('Return (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))

axes[1].fill_between(ew_cum.index, ew_cum.values * 100, alpha=0.3, color='steelblue')
axes[1].plot(ew_cum.index, ew_cum.values * 100, color='steelblue', lw=2)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Equal-Weighted Portfolio: Cumulative Return')
axes[1].set_ylabel('Cumulative Return (%)')

plt.tight_layout(); plt.show()

In [ ]:
# Sector cumulative returns
sec_monthly = df.groupby(['month','sector'])['return'].mean().unstack()
sec_cum = (1 + sec_monthly).cumprod() - 1

# Figure 10 — Cumulative return by sector (line chart)
fig, ax = plt.subplots(figsize=(12, 6))
for col, color in zip(sec_cum.columns, sns.color_palette('tab10', len(sec_cum.columns))):
    ax.plot(sec_cum.index, sec_cum[col] * 100, lw=2, label=col, color=color)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title('Cumulative Return by Sector (2022–2024)')
ax.set_ylabel('Cumulative Return (%)')
ax.legend(fontsize=8, ncol=2, loc='upper left')
plt.tight_layout(); plt.show()

In [ ]:
# Figure 11 — Stacked area chart: monthly EW return contributions by sector
# (sector contribution = sector weight × sector return; here equal-weighted across sectors)
sec_w = sec_monthly.copy()
fig, ax = plt.subplots(figsize=(12, 5))
colors = sns.color_palette('tab10', sec_w.shape[1])
ax.stackplot(sec_w.index,
             [sec_w[c].fillna(0) * 100 for c in sec_w.columns],
             labels=sec_w.columns, colors=colors, alpha=0.8)
ax.axhline(0, color='black', lw=1)
ax.set_title('Stacked Monthly Returns by Sector')
ax.set_ylabel('Return (%)')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

In [ ]:
# Figure 12 — Heatmap: individual stock monthly returns
pivot = df.pivot(index='ticker', columns='month', values='return') * 100
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot, cmap='RdYlGn', center=0, linewidths=0.2,
            cbar_kws={'label': 'Monthly Return (%)'}, ax=ax,
            xticklabels=4)
ax.set_title('Monthly Returns by Ticker (%)')
ax.set_xlabel('Month'); ax.set_ylabel('')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

---
## 5. Rolling Returns

In [ ]:
# Figure 13 — Rolling 6-month average return by sector
fig, ax = plt.subplots(figsize=(12, 6))
for col, color in zip(sec_monthly.columns, sns.color_palette('tab10', len(sec_monthly.columns))):
    roll = sec_monthly[col].rolling(6).mean() * 100
    ax.plot(roll.index, roll.values, lw=2, label=col, color=color)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title('Rolling 6-Month Average Monthly Return by Sector')
ax.set_ylabel('Return (%)')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

---
## 6. Quintile Portfolio Analysis

Each month we sort stocks into 5 quintiles (Q1 = bottom, Q5 = top) by a signal measured at the *start* of the month, then compute the equal-weighted return of each quintile.

In [ ]:
def quintile_portfolios(df, signal, label=None):
    """Sort into quintiles each month; return monthly quintile returns and summary."""
    label = label or signal
    sub = df.dropna(subset=[signal, 'return']).copy()
    sub['q'] = sub.groupby('month')[signal].transform(
        lambda x: pd.qcut(x, 5, labels=['Q1','Q2','Q3','Q4','Q5'])
    )
    monthly = sub.groupby(['month','q'])['return'].mean().unstack()
    monthly.columns.name = label
    monthly['Spread (Q5−Q1)'] = monthly['Q5'] - monthly['Q1']
    summary = monthly.mean().rename('Mean Return').to_frame()
    summary['t-stat'] = monthly.apply(lambda c: stats.ttest_1samp(c.dropna(), 0).statistic)
    summary['Ann. Return'] = (1 + summary['Mean Return'])**12 - 1
    return monthly, summary.round(4)

In [ ]:
# ── Momentum quintiles ────────────────────────────────────────────────
mom_monthly, mom_summary = quintile_portfolios(df, 'momentum', 'Momentum')

# Table 4 — Momentum quintile summary
print('=== Momentum Quintile Portfolios ===')
mom_summary

In [ ]:
# Figure 14 — Momentum quintile cumulative returns
mom_cum = (1 + mom_monthly.drop(columns='Spread (Q5−Q1)')).cumprod() - 1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#d62728','#ff7f0e','#2ca02c','#1f77b4','#9467bd']
for col, c in zip(mom_cum.columns, colors):
    axes[0].plot(mom_cum.index, mom_cum[col]*100, lw=2, label=col, color=c)
axes[0].axhline(0, color='black', lw=0.6, ls='--')
axes[0].set_title('Momentum: Cumulative Returns by Quintile')
axes[0].set_ylabel('Cumulative Return (%)')
axes[0].legend()

spread_cum = (1 + mom_monthly['Spread (Q5−Q1)']).cumprod() - 1
axes[1].fill_between(spread_cum.index, spread_cum*100,
                     color='steelblue', alpha=0.35)
axes[1].plot(spread_cum.index, spread_cum*100, color='steelblue', lw=2)
axes[1].axhline(0, color='black', lw=0.8, ls='--')
axes[1].set_title('Momentum: Q5−Q1 Spread (Cumulative)')
axes[1].set_ylabel('Cumulative Return (%)')

plt.tight_layout(); plt.show()

In [ ]:
# ── Lagged return (short-term reversal) ──────────────────────────────
rev_monthly, rev_summary = quintile_portfolios(df, 'lagged_return', 'Lagged Return')

# Table 5 — Lagged return quintile summary
print('=== Lagged Return Quintile Portfolios ===')
rev_summary

In [ ]:
# Figure 15 — Lagged return quintile cumulative returns
rev_cum = (1 + rev_monthly.drop(columns='Spread (Q5−Q1)')).cumprod() - 1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for col, c in zip(rev_cum.columns, colors):
    axes[0].plot(rev_cum.index, rev_cum[col]*100, lw=2, label=col, color=c)
axes[0].axhline(0, color='black', lw=0.6, ls='--')
axes[0].set_title('Lagged Return: Cumulative Returns by Quintile')
axes[0].set_ylabel('Cumulative Return (%)')
axes[0].legend()

spread = (1 + rev_monthly['Spread (Q5−Q1)']).cumprod() - 1
axes[1].fill_between(spread.index, spread*100, color='crimson', alpha=0.35)
axes[1].plot(spread.index, spread*100, color='crimson', lw=2)
axes[1].axhline(0, color='black', lw=0.8, ls='--')
axes[1].set_title('Lagged Return: Q5−Q1 Spread (Cumulative)')
axes[1].set_ylabel('Cumulative Return (%)')

plt.tight_layout(); plt.show()

In [ ]:
# ── Size (market cap) quintiles ──────────────────────────────────────
sz_monthly, sz_summary = quintile_portfolios(df, 'marketcap_billions', 'Size')

# Table 6 — Size quintile summary
print('=== Size Quintile Portfolios ===')
sz_summary

In [ ]:
# ── Value (P/B) quintiles ─────────────────────────────────────────────
val_monthly, val_summary = quintile_portfolios(df, 'pb', 'P/B Ratio')

# Table 7 — Value quintile summary
print('=== P/B Quintile Portfolios (Q1 = cheapest) ===')
val_summary

In [ ]:
# Figure 16 — Size and P/B quintile bar charts side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, summary, title in [
    (axes[0], sz_summary, 'Size (Market Cap) Quintile Returns'),
    (axes[1], val_summary, 'P/B Quintile Returns (Q1 = cheapest)')
]:
    qs = ['Q1','Q2','Q3','Q4','Q5']
    means = summary.loc[qs, 'Mean Return'] * 100
    cs = ['#d62728' if v < 0 else '#1f77b4' for v in means]
    ax.bar(qs, means, color=cs, edgecolor='white')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(title)
    ax.set_ylabel('Mean Monthly Return (%)')

plt.tight_layout(); plt.show()

In [ ]:
# Table 8 — Factor spread table (Q5−Q1 across all signals)
signals = {'Momentum': mom_summary, 'Lagged Return': rev_summary,
           'Size': sz_summary, 'P/B': val_summary}
spread_rows = []
for name, summ in signals.items():
    row = summ.loc['Spread (Q5−Q1)']
    spread_rows.append({'Signal': name,
                        'Mean Monthly Spread': row['Mean Return'],
                        't-stat': row['t-stat'],
                        'Ann. Spread': row['Ann. Return']})
spread_table = pd.DataFrame(spread_rows).set_index('Signal').round(4)
print('=== Factor Spread Summary (Q5 minus Q1) ===')
spread_table

In [ ]:
# Figure 17 — Bar chart of factor spreads
fig, ax = plt.subplots(figsize=(8, 4))
spreads = spread_table['Mean Monthly Spread'] * 100
cs = ['#2ca02c' if v > 0 else '#d62728' for v in spreads]
ax.bar(spreads.index, spreads.values, color=cs, edgecolor='white', width=0.5)
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Q5−Q1 Monthly Spread by Signal')
ax.set_ylabel('Mean Monthly Return (%)')
for i, (v, t) in enumerate(zip(spreads.values, spread_table['t-stat'])):
    ax.text(i, v + 0.05 * np.sign(v), f't={t:.2f}',
            ha='center', va='bottom' if v > 0 else 'top', fontsize=9)
plt.tight_layout(); plt.show()

---
## 7. Signal Correlations and Pairplots

In [ ]:
# Figure 18 — Correlation heatmap of signals and return
corr_cols = ['return','momentum','lagged_return','marketcap_billions',
             'pb','roe','grossmargin','gp_to_assets','asset_growth']
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Signal Correlation Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# Figure 19 — Pairplot: return, momentum, lagged_return, pb
pair_cols = ['return','momentum','lagged_return','pb']
pair_df = df[pair_cols + ['sector']].dropna()
g = sns.pairplot(pair_df, vars=pair_cols, hue='sector',
                 plot_kws={'alpha': 0.4, 's': 15},
                 diag_kind='kde', corner=True,
                 palette='tab10')
g.figure.suptitle('Pairplot: Return, Momentum, Lagged Return, P/B', y=1.02)
plt.show()

In [ ]:
# Figure 20 — Pairplot: fundamental signals
fund_cols = ['return','roe','grossmargin','gp_to_assets','asset_growth']
fund_df = df[fund_cols + ['sector']].dropna()
g2 = sns.pairplot(fund_df, vars=fund_cols, hue='sector',
                  plot_kws={'alpha': 0.4, 's': 15},
                  diag_kind='kde', corner=True,
                  palette='tab10')
g2.figure.suptitle('Pairplot: Return and Fundamental Signals', y=1.02)
plt.show()

---
## 8. Scatter Plots and Regression Lines (regplots)

In [ ]:
# Figure 21 — regplot: momentum vs return (all months)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, x_col, title in [
    (axes[0], 'momentum',     'Momentum vs. Return'),
    (axes[1], 'lagged_return','Lagged Return vs. Return'),
]:
    sns.regplot(data=df, x=x_col, y='return', ax=ax,
                scatter_kws={'alpha': 0.25, 's': 15, 'color': 'steelblue'},
                line_kws={'color': 'crimson', 'lw': 2})
    ax.set_title(title)
    ax.set_xlabel(x_col); ax.set_ylabel('Monthly Return')

plt.tight_layout(); plt.show()

In [ ]:
# Figure 22 — regplot: pb and roe vs return
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, x_col, title in [
    (axes[0], 'pb',  'P/B Ratio vs. Return'),
    (axes[1], 'roe', 'ROE vs. Return'),
]:
    sns.regplot(data=df, x=x_col, y='return', ax=ax,
                scatter_kws={'alpha': 0.25, 's': 15, 'color': 'seagreen'},
                line_kws={'color': 'darkorange', 'lw': 2})
    ax.set_title(title)
    ax.set_xlabel(x_col); ax.set_ylabel('Monthly Return')

plt.tight_layout(); plt.show()

In [ ]:
# Figure 23 — lmplot: momentum vs return, faceted by sector
lm_df = df[['momentum','return','sector']].dropna()
g = sns.lmplot(data=lm_df, x='momentum', y='return', col='sector',
               col_wrap=4, height=3.5, aspect=1.1,
               scatter_kws={'alpha': 0.3, 's': 12},
               line_kws={'color': 'crimson', 'lw': 1.5})
g.set_titles('{col_name}')
g.figure.suptitle('Momentum vs. Return by Sector', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

---
## 9. OLS Regressions

In [ ]:
# Table 9 — OLS: return on momentum and lagged_return (pooled)
ols1 = smf.ols('Q("return") ~ momentum + lagged_return',
               data=df.dropna(subset=['return','momentum','lagged_return'])).fit(
               cov_type='HC3'
)
print(ols1.summary2())

In [ ]:
# OLS: return on multiple signals (fundamentals), sector fixed effects
sig_cols = ['momentum','lagged_return','pb','roe','grossmargin','gp_to_assets','asset_growth']
fe_df = df[['return','sector'] + sig_cols].dropna()
fe_df['sector_fe'] = fe_df['sector'].astype('category')
formula = 'Q("return") ~ ' + ' + '.join(sig_cols) + ' + C(sector_fe)'
ols2 = smf.ols(formula, data=fe_df).fit(cov_type='HC3')
print(ols2.summary2())

In [ ]:
# Figure 24 — Coefficient plot for the multi-signal regression
coef = ols2.params.filter(sig_cols)
ci   = ols2.conf_int().loc[sig_cols]

fig, ax = plt.subplots(figsize=(8, 5))
y_pos = range(len(coef))
ax.barh(y_pos, coef.values,
        xerr=[(coef.values - ci[0].values), (ci[1].values - coef.values)],
        color=['#d62728' if v < 0 else '#1f77b4' for v in coef.values],
        capsize=4, edgecolor='white', height=0.5)
ax.set_yticks(y_pos); ax.set_yticklabels(coef.index)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('OLS Coefficients: Return on Signals + Sector FE (±95% CI)')
ax.set_xlabel('Coefficient')
plt.tight_layout(); plt.show()

In [ ]:
# Fama-MacBeth: cross-sectional regression each month, then average
fm_signals = ['momentum','lagged_return','pb']
monthly_coefs = []
for month, grp in df.dropna(subset=['return'] + fm_signals).groupby('month'):
    if len(grp) < 10:
        continue
    res = smf.ols('Q("return") ~ ' + ' + '.join(fm_signals), data=grp).fit()
    row = res.params.to_dict()
    row['month'] = month
    monthly_coefs.append(row)

fm_df = pd.DataFrame(monthly_coefs).set_index('month')

# Table 10 — Fama-MacBeth average coefficients
fm_summary = pd.DataFrame({
    'Mean Coef': fm_df.mean(),
    'Std':       fm_df.std(),
    't-stat':    fm_df.mean() / (fm_df.std() / np.sqrt(len(fm_df)))
}).round(5)
print('=== Fama-MacBeth Regression Summary ===')
fm_summary

In [ ]:
# Figure 25 — Fama-MacBeth monthly coefficients over time
fig, axes = plt.subplots(len(fm_signals) + 1, 1,
                         figsize=(12, 3 * (len(fm_signals) + 1)), sharex=True)
for ax, col in zip(axes, fm_df.columns):
    ax.bar(fm_df.index, fm_df[col],
           color=['#d62728' if v < 0 else '#1f77b4' for v in fm_df[col]],
           width=20)
    ax.axhline(0, color='black', lw=0.6)
    ax.axhline(fm_df[col].mean(), color='crimson', lw=1.5, ls='--',
               label=f'Mean={fm_df[col].mean():.4f}')
    ax.set_title(f'Monthly Coefficient: {col}')
    ax.legend(fontsize=8)
plt.suptitle('Fama-MacBeth Monthly Coefficients', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

---
## 10. Filtering Examples

In [ ]:
# Split into large-cap and small-cap halves by median market cap within each month
df['cap_group'] = df.groupby('month')['marketcap_billions'].transform(
    lambda x: pd.qcut(x, 2, labels=['Small Cap','Large Cap'])
)
cap_monthly = df.groupby(['month','cap_group'])['return'].mean().unstack()
cap_cum = (1 + cap_monthly).cumprod() - 1

# Figure 26 — Large cap vs small cap cumulative return
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(cap_cum.index, cap_cum['Large Cap']*100, lw=2,
        color='steelblue', label='Large Cap')
ax.plot(cap_cum.index, cap_cum['Small Cap']*100, lw=2,
        color='crimson', label='Small Cap', ls='--')
ax.fill_between(cap_cum.index,
                cap_cum['Large Cap']*100,
                cap_cum['Small Cap']*100,
                alpha=0.15, color='gray')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title('Large Cap vs. Small Cap: Cumulative Return')
ax.set_ylabel('Cumulative Return (%)')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Filter: only Technology and Healthcare stocks
tech_health = df[df['sector'].isin(['Technology','Healthcare'])].copy()

# Figure 27 — Box plots of returns: Technology vs Healthcare
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=tech_health, x='ticker', y='return',
            hue='sector', dodge=False,
            palette={'Technology':'steelblue','Healthcare':'seagreen'},
            width=0.6, fliersize=3, ax=ax)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title('Monthly Returns: Technology and Healthcare Stocks')
ax.set_ylabel('Monthly Return')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Sector')
plt.tight_layout(); plt.show()

In [ ]:
# Filter: 2022 bear market vs 2023-2024 recovery
df['period'] = df['year'].map({2022:'2022 (Bear)', 2023:'2023 (Recovery)', 2024:'2024'})

# Figure 28 — Density by period
fig, ax = plt.subplots(figsize=(10, 5))
palette = {'2022 (Bear)':'#d62728','2023 (Recovery)':'#2ca02c','2024':'#1f77b4'}
for period, color in palette.items():
    vals = df.loc[df['period'] == period, 'return'].dropna()
    kde = stats.gaussian_kde(vals)
    x = np.linspace(-0.35, 0.40, 300)
    ax.fill_between(x, kde(x), alpha=0.25, color=color, label=period)
    ax.plot(x, kde(x), color=color, lw=2)
ax.axvline(0, color='black', lw=0.8, ls='--')
ax.set_title('Return Distribution by Market Period')
ax.set_xlabel('Monthly Return'); ax.set_ylabel('Density')
ax.legend()
plt.tight_layout(); plt.show()

---
## 11. Top and Bottom Performers

In [ ]:
# Cumulative return by ticker over the full period
ticker_cum = (df.groupby(['ticker','month'])['return'].first()
                .unstack().T
                .apply(lambda c: (1 + c).cumprod() - 1))
final_ret = ticker_cum.iloc[-1].sort_values(ascending=False)

# Table 11 — Top 10 and bottom 10 cumulative returners
merged = final_ret.rename('Cumulative Return').to_frame()
merged['Sector'] = df.drop_duplicates('ticker').set_index('ticker')['sector']
print('=== Top 10 Performers (Jan 2022 – Dec 2024) ===')
display(merged.head(10).style.format({'Cumulative Return': '{:.1%}'}))
print('\n=== Bottom 10 Performers ===')
merged.tail(10).style.format({'Cumulative Return': '{:.1%}'})

In [ ]:
# Figure 29 — Cumulative returns: top 5 and bottom 5 tickers
top5    = final_ret.head(5).index.tolist()
bottom5 = final_ret.tail(5).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, group, title, cmap in [
    (axes[0], top5,    'Top 5 Cumulative Performers',    'Blues_r'),
    (axes[1], bottom5, 'Bottom 5 Cumulative Performers', 'Reds_r'),
]:
    colors = sns.color_palette(cmap, 5)
    for tick, col in zip(group, colors):
        s = ticker_cum[tick]
        ax.plot(s.index, s.values * 100, lw=2, label=tick, color=col)
    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_title(title); ax.set_ylabel('Cumulative Return (%)')
    ax.legend()

plt.tight_layout(); plt.show()

In [ ]:
# Figure 30 — Horizontal bar chart: all tickers ranked by cumulative return
fig, ax = plt.subplots(figsize=(9, 10))
sector_colors = dict(zip(df['sector'].unique(),
                         sns.color_palette('tab10', df['sector'].nunique())))
bar_colors = [sector_colors[merged.loc[t,'Sector']] for t in final_ret.index]
ax.barh(final_ret.index, final_ret.values * 100,
        color=bar_colors, edgecolor='white', height=0.7)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Cumulative Return (%)')
ax.set_title('3-Year Cumulative Return by Ticker (colored by sector)')
# legend
from matplotlib.patches import Patch
handles = [Patch(facecolor=c, label=s) for s, c in sector_colors.items()]
ax.legend(handles=handles, fontsize=8, loc='lower right')
plt.tight_layout(); plt.show()

---
## 12. Momentum Distribution by Sector

In [ ]:
# Figure 31 — Boxplot of momentum by sector
mom_order = df.groupby('sector')['momentum'].median().sort_values().index
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df.dropna(subset=['momentum']), x='sector', y='momentum',
            order=mom_order, palette='tab10', width=0.5, fliersize=3, ax=ax)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel(''); ax.set_ylabel('12-Month Momentum')
ax.set_title('Momentum Distribution by Sector')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.show()

In [ ]:
# Figure 32 — Stacked bar: count of stocks per sector per year
cnt = df.groupby(['year','sector'])['ticker'].nunique().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(9, 5))
cnt.plot(kind='bar', stacked=True, ax=ax,
         colormap='tab10', edgecolor='white')
ax.set_title('Number of Stocks per Sector per Year')
ax.set_xlabel('Year'); ax.set_ylabel('Number of Stocks')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Sector', fontsize=8, bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()